In [1]:
# %% [markdown]
# Notebook ETL – leitura em batch, validações e write-back

In [2]:
import os, sys, importlib
project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)
logs_dir = os.path.join(project_root, 'logs')
if logs_dir not in sys.path:
    sys.path.insert(0, logs_dir)
import logging_setup
importlib.reload(logging_setup)
from logging_setup import get_logger  # importa e já faz setup_logging
logger = get_logger(__name__)

In [3]:
# %% [code]
import os
import sys
from dotenv import load_dotenv

project_root = os.getcwd()  # supondo que o notebook esteja em /home/debrito/Documentos/etl_debrito
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# ── Carrega variáveis de ambiente de .env (opcional) ───────────────────────
load_dotenv()

False

In [4]:
#2 %% [code]
import math
import numpy as np
from typing import Any

def _to_json_safe(x: Any) -> Any:
    """
    Internal helper to convert arbitrary objects into JSON-safe primitives.
    """
    if x is None:
        return None
    if isinstance(x, (int, str, bool)):
        return x
    if isinstance(x, float):
        return None if math.isnan(x) or math.isinf(x) else x
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return None if (math.isnan(x) or math.isinf(x)) else float(x)
    if isinstance(x, (list, tuple, set)):
        return [_to_json_safe(item) for item in x]
    if isinstance(x, dict):
        return {k: _to_json_safe(v) for k, v in x.items()}
    # Fallback: stringify anything else
    return str(x)

def json_safe(obj: Any) -> Any:
    """
    Recursively converts `obj` into structures 100% serializable to JSON.
    """
    return _to_json_safe(obj)

In [5]:
#3 %% [code]
# Flags de gravação (ajuste conforme necessidade)
WRITE_BACK_ORIGIN = True   # grava na aba-origem (meta*, tiktok*, …)
WRITE_BACK_DEST   = True   # grava nas abas-modelo (modelo*)
DRY_RUN_DEST      = False  # True = simula write-back destino

# Credenciais e identificador da planilha
CREDS_PATH     = os.getenv("GOOGLE_CREDS_PATH", "creds.json")
SPREADSHEET_ID = os.getenv(
    "GOOGLE_SHEET_ID",
    "1DazUQxspLgT0utOFHcTINbFngXw7Fq0LOq6v4lRGixg"
)

# Abas de origem a processar, agrupadas por plataforma
SHEET_NAMES = [
    # Meta (Facebook / Instagram)
    "metaGeral",
    "metaIdade",
    "metaGenero",
    "metaRegiao",
    "metaAlcance",
    # TikTok
    "tiktokGeral",
    "tiktokIdade",
    "tiktokGenero",
    "tiktokRegiao",
    "tiktokAlcance",
    # Pinterest
    "pinterestGeral",
    "pinterestGenero",
    "pinterestIdade",
    "pinterestRegiao",
    "pinterestAlcance",
    # LinkedIn
    "linkedinGeral",
    "linkedinRegiao",
    "linkedinAlcance",
    # Google Analytics
    "GAGeral",
]

In [6]:
#4
# %% [code]
import importlib

# módulos principais e helpers recarregados
import extract.sheets_fetcher               as sf_mod
import treat.treat_pipeline                 as tp_mod
import treat.platforms                      as platforms_mod
import treat.platforms.linkedin             as linkedin_mod
import treat.platforms.tiktok               as tiktok_mod
import treat.platforms.pinterest            as pinterest_mod
import treat.platforms.meta                 as meta_mod
import treat.platforms.ga                   as ga_mod
import load.origin_writer                   as ow_mod
import load.dest_writer                     as dw_mod
import treat.utils.renomeacoes              as rn_mod
import treat.utils.preview_links            as prev_mod
import treat.utils.atribuicoes_via_lookup   as atrib_mod
import treat.utils.substitute_origin_values as sub_mod
import treat.utils.preprocess_utils         as pre_mod
import treat.utils.geo_normalize            as geo_mod

# hot-reload de todos os módulos alterados durante o desenvolvimento
for m in (
    sf_mod,
    tp_mod,
    platforms_mod,
    linkedin_mod,
    tiktok_mod,
    pinterest_mod,
    meta_mod,
    ga_mod,
    ow_mod,
    dw_mod,
    rn_mod,
    prev_mod,
    atrib_mod,
    sub_mod,
    pre_mod,
    geo_mod,
):
    importlib.reload(m)

In [7]:
# %% [code]
# Cell 5: definição do helper run_etl_for_sheet
import pandas as pd
import gc
import json
from pprint import pp
from typing import Dict

from logging_setup import get_logger
logger = get_logger(__name__)

from extract.sheets_fetcher import SheetsFetcher
from treat.treat_pipeline import TreatPipeline
from treat.utils.renomeacoes import renomeacao_geral, renomear_colunas_origem_para_modelo
from treat.utils.campos_calculados import calcular_engajamento_total, gerar_id
from load.origin_writer import write_back_origin
from load.dest_writer import write_back_for_sheet

# Instância única do fetcher (retry/backoff/cache interno)
fetcher = SheetsFetcher(
    spreadsheet_id=SPREADSHEET_ID,
    creds_path=CREDS_PATH,
)

def run_etl_for_sheet(
    *,
    sheet: str,
    wb_origin_flag: bool,
    wb_dest_flag: bool,
    dry_run_dest: bool,
    preloaded_raw: pd.DataFrame,
) -> Dict[str, pd.DataFrame | dict]:
    """
    Executa o fluxo completo para uma aba e devolve:
      { "dest": DataFrame destino (ou vazio), "taxo": relatório de taxonomia }
    """
    # 1) Dados brutos já carregados
    df_raw = preloaded_raw

    # 2) Tratamento via pipeline
    pipeline = TreatPipeline(
        creds_path=CREDS_PATH,
        spreadsheet_id=SPREADSHEET_ID,
        sheet_name=sheet,
        mapping_renomeacao=renomeacao_geral,
        write_back=wb_origin_flag,
    )
    df_ok = pipeline.run(df_raw)

    # 3) Relatório de taxonomia
    taxo_report = getattr(pipeline, "_last_taxo_report", {})
    pp(json.dumps(taxo_report, default=str), width=120)

    # 4) Write-back na aba de origem (apenas quando não for Pinterest demográfico)
    is_pinterest_dim = sheet.lower() in {
        "pinterestgenero", "pinterestidade", "pinterestregiao"
    }

    if not is_pinterest_dim:
        # grava correções de pré-processamento in-place
        _ = write_back_origin(
            df_raw        = df_raw,
            df_ok         = df_ok,
            creds_path    = CREDS_PATH,
            spreadsheet_id= SPREADSHEET_ID,
            sheet_name    = sheet,
            write_back    = wb_origin_flag,
            dry_run       = not wb_origin_flag,
        )
    else:
        logger.debug(
            "🔸 %s: pulando write-back de origem (já feito dentro de pipeline)",
            sheet
        )

    # 5) Preparar DataFrame de destino (modelo)
    df_model = renomear_colunas_origem_para_modelo(df_ok, renomeacao_geral)
    df_model = calcular_engajamento_total(df_model)
    df_model["ID"] = df_model.apply(gerar_id, axis=1)

    # 6) Write-back de destino  ──────────────────────────────────────────────
    if sheet.lower().startswith("ga"):
        logger.info("🔸 %s: write-back de destino ignorado (Google Analytics)", sheet)
        df_dest = pd.DataFrame()      # retorna DataFrame vazio
    else:
        df_dest = write_back_for_sheet(
            df_model,
            sheet_name     = sheet,
            creds_path     = CREDS_PATH,
            spreadsheet_id = SPREADSHEET_ID,
            write_back     = wb_dest_flag,
            dry_run        = dry_run_dest,
        )
        if df_dest is None:
            df_dest = pd.DataFrame()

    # 7) Retorno
    return {"dest": df_dest, "taxo": taxo_report}

In [ ]:
# %% [code]
%xmode verbose
from contextlib import suppress
import gc
import pandas as pd
from tqdm.auto import tqdm

from logs.logging_setup import get_logger
from load.dest_writer import prefetch_meta

logger = get_logger(__name__)

def process_sheets(
    fetcher,
    sheet_names: list[str],
    spreadsheet_id: str,
    write_origin: bool,
    write_dest: bool,
    dry_run: bool
) -> dict[str, dict[str, object]]:
    """
    Lê todas as abas, faz prefetch de metadados e executa o ETL em cada aba.
    Retorna um dict com os resultados por aba.
    """
    logger.info("🔄 Iniciando processamento de abas")

    # 1) Leitura batch das abas
    raw_map = fetcher.get(sheet_names)

    # 2) Copiando para evitar mutação in-place e validando tipos
    all_raw: dict[str, pd.DataFrame] = {}
    for name, df in raw_map.items():
        if isinstance(df, pd.DataFrame):
            all_raw[name] = df.copy()
        else:
            logger.error(f"Aba '{name}' não é um DataFrame (tipo={type(df)}); será ignorada.")

    # 3) Debug das colunas originais
    for name, df in all_raw.items():
        logger.debug(f"Aba '{name}' colunas originais: {df.columns.tolist()}")

    # 4) Prefetch de headers e IDs das abas-modelo
    prefetch_meta(fetcher, spreadsheet_id)
    logger.info("📥 Prefetch meta concluído – iniciando ETL por aba")

    # 5) Processamento aba a aba
    results: dict[str, dict[str, object]] = {}
    for sheet in tqdm(sheet_names, desc="Processando abas"):
        try:
            df_raw = all_raw.get(sheet)
            if df_raw is None:
                logger.warning(f"Aba '{sheet}' não carregada; pulando ETL.")
                continue

            # Log específico para GA
            if sheet.lower().startswith("ga"):
                logger.info(f"🔸 {sheet}: apenas write-back de origem; destino será ignorado")

            out = run_etl_for_sheet(
                sheet=sheet,
                wb_origin_flag=write_origin,
                wb_dest_flag=write_dest,
                dry_run_dest=dry_run,
                preloaded_raw=df_raw,
            )

            results[sheet] = {
                "dest": out.get("dest"),
                "taxo": out.get("taxo")
            }
            logger.debug(f"Aba '{sheet}' processada com sucesso")

        except Exception as e:
            logger.exception(f"Erro ao processar aba '{sheet}': {e}")

        finally:
            # Garantir liberação de memória mesmo em caso de erro
            gc.collect()

    logger.info("✅ Processamento de todas as abas concluído")
    return results

# Chamando a função
results = process_sheets(
    fetcher=fetcher,
    sheet_names=SHEET_NAMES,
    spreadsheet_id=SPREADSHEET_ID,
    write_origin=WRITE_BACK_ORIGIN,
    write_dest=WRITE_BACK_DEST,
    dry_run=DRY_RUN_DEST,
)


Exception reporting mode: Verbose
19:10:35 INFO __main__ › 🔄 Iniciando processamento de abas
19:10:35 INFO extract.sheets_fetcher › 🔄 batchGet tentativa para ranges: ['metaGeral!A:ZZ', 'metaIdade!A:ZZ', 'metaGenero!A:ZZ', 'metaRegiao!A:ZZ', 'metaAlcance!A:ZZ', 'tiktokGeral!A:ZZ', 'tiktokIdade!A:ZZ', 'tiktokGenero!A:ZZ', 'tiktokRegiao!A:ZZ', 'tiktokAlcance!A:ZZ', 'pinterestGeral!A:ZZ', 'pinterestGenero!A:ZZ', 'pinterestIdade!A:ZZ', 'pinterestRegiao!A:ZZ', 'pinterestAlcance!A:ZZ', 'linkedinGeral!A:ZZ', 'linkedinRegiao!A:ZZ', 'linkedinAlcance!A:ZZ', 'GAGeral!A:ZZ']
19:10:38 INFO extract.sheets_fetcher › 🔍 range completo retornado pela API: metaGeral!A1:AB615
19:10:38 INFO extract.sheets_fetcher › 🔍 range completo retornado pela API: metaIdade!A1:Q2068
19:10:38 INFO extract.sheets_fetcher › 🔍 range completo retornado pela API: metaGenero!A1:T928
19:10:38 INFO extract.sheets_fetcher › 🔍 range completo retornado pela API: metaRegiao!A1:Q10390
19:10:38 INFO extract.sheets_fetcher › 🔍 range co

Processando abas:   0%|          | 0/19 [00:00<?, ?it/s]

19:10:43 WARNING treat.utils.validations › [Validação] 34 valor(es) de 'ad_name' fora da BI_PARAMETRIZAÇÃO (taxonomy_ad_name): ['2025_3_BR_CARD_NAT_ACAO_DBT_SBRAE_2025_EMP_FEM0028', '2025_3_BR_CARD_SILVANA_ACAO_DBT_SBRAE_2025_EMP_FEM0044', '2025_3_BR_STORIES_CINTIA_ACAO_DBT_SBRAE_2025_EMP_FEM0021', '2025_3_BR_STORIES_MANIFESTO_ACAO_DBT_SBRAE_2025_EMP_FEM0061', '2025_3_BR_STORIES_NAT_ACAO_DBT_SBRAE_2025_EMP_FEM0041', '2025_3_BR_STORIES_SILVANA_ACAO_DBT_SBRAE_2025_EMP_FEM0057', '2025_3_BR_VÍDEO_MANIFESTO_ACAO_DBT_SBRAE_2025_EMP_FEM0056', '2025_3_BR_VÍDEO_MANIFESTO_ACAO_DBT_SBRAE_2025_EMP_FEM0064', '2025_3_BR_CARD_CERRADO_COMO_SE_INSCREVER_ACAO_DBT_SBRAE_2025_CER_PAN0150', '2025_3_BR_CARD_CERRADO_DIA_DAS_INSCRIÇÕES_ACAO_DBT_SBRAE_2025_CER_PAN0146', '2025_3_BR_CARD_CERRADO_MOTIVOS_PARA_INSCREVER_ACAO_DBT_SBRAE_2025_CER_PAN0152', '2025_3_BR_CARD_CERRADO_PANTANAL_COMO_SE_INSCREVER_ACAO_DBT_SBRAE_2025_CER_PAN0153', '2025_3_BR_CARD_CERRADO_QUEM_PODE_PARTICIPAR_ACAO_DBT_SBRAE_2025_CER_PAN0142',

In [9]:
# %% [code]
# Cell 7: Validação de consistência de datas entre modelos e estatísticas de uso
from logging_setup import get_logger
logger = get_logger(__name__)

from treat.treat_pipeline import BIParamLookup
from treat.utils.validations import validate_consistent_dates_across_models

import gspread
import google.auth

# ── 1) Extrair apenas os DataFrames de destino ─────────────────────────────
dest_dfs = {sheet: info["dest"] for sheet, info in results.items()}

# ── 2) Validar consistência de datas ───────────────────────────────────────
logger.info("🔍 Validando consistência de datas entre modelos …")
df_inconsistencies = validate_consistent_dates_across_models(dest_dfs)

if df_inconsistencies is not None and not df_inconsistencies.empty:
    logger.warning("💥 Inconsistências encontradas:")
    display(df_inconsistencies)
else:
    logger.info("✅ Nenhuma divergência de start/end entre modelos.")

# ── 3) Limpar caches se necessário ─────────────────────────────────────────
# Limpa cache de leituras em lote (SheetsFetcher já instanciado como `fetcher`)
fetcher.refresh(SHEET_NAMES)
# Limpa cache da parametrização BI em memória
BIParamLookup._df = None
BIParamLookup._last_load = 0.0

# ── 4) Estatísticas de uso das planilhas ───────────────────────────────────
creds, _ = google.auth.load_credentials_from_file(CREDS_PATH, scopes=[
    "https://www.googleapis.com/auth/spreadsheets.readonly"
])
gc = gspread.authorize(creds)
ss = gc.open_by_key(SPREADSHEET_ID)

stats = []
for ws in ss.worksheets():
    rows = ws.row_count
    cols = ws.col_count
    cells = rows * cols
    stats.append((cells, ws.title, rows, cols))

stats.sort(reverse=True)          # maiores primeiro
logger.info("📊 Top 10 abas que mais ocupam células:")
for cells, title, rows, cols in stats[:10]:
    logger.info(f"  • {title}: {rows}×{cols} = {cells:,} células")

19:12:08 INFO __main__ › 🔍 Validando consistência de datas entre modelos …
19:12:08 INFO treat.utils.validations › ✅ Nenhuma divergência de start/end entre modelos.
19:12:08 INFO __main__ › ✅ Nenhuma divergência de start/end entre modelos.
19:12:08 INFO extract.sheets_fetcher › 🔄 batchGet tentativa para ranges: ['metaGeral!A:ZZ', 'metaIdade!A:ZZ', 'metaGenero!A:ZZ', 'metaRegiao!A:ZZ', 'metaAlcance!A:ZZ', 'tiktokGeral!A:ZZ', 'tiktokIdade!A:ZZ', 'tiktokGenero!A:ZZ', 'tiktokRegiao!A:ZZ', 'tiktokAlcance!A:ZZ', 'pinterestGeral!A:ZZ', 'pinterestGenero!A:ZZ', 'pinterestIdade!A:ZZ', 'pinterestRegiao!A:ZZ', 'pinterestAlcance!A:ZZ', 'linkedinGeral!A:ZZ', 'linkedinRegiao!A:ZZ', 'linkedinAlcance!A:ZZ', 'GAGeral!A:ZZ']
19:12:10 INFO extract.sheets_fetcher › 🔍 range completo retornado pela API: metaGeral!A1:AB615
19:12:10 INFO extract.sheets_fetcher › 🔍 range completo retornado pela API: metaIdade!A1:Q2068
19:12:10 INFO extract.sheets_fetcher › 🔍 range completo retornado pela API: metaGenero!A1:T928

In [10]:
#8
# %% [code]
from treat.treat_pipeline import BIParamLookup

# — Limpa cache de leituras em lote (SheetsFetcher já instanciado como `fetcher` em células anteriores)
fetcher.refresh(SHEET_NAMES)

# — Limpa cache da parametrização BI em memória
BIParamLookup._df = None
BIParamLookup._last_load = 0.0

19:12:11 INFO extract.sheets_fetcher › 🔄 batchGet tentativa para ranges: ['metaGeral!A:ZZ', 'metaIdade!A:ZZ', 'metaGenero!A:ZZ', 'metaRegiao!A:ZZ', 'metaAlcance!A:ZZ', 'tiktokGeral!A:ZZ', 'tiktokIdade!A:ZZ', 'tiktokGenero!A:ZZ', 'tiktokRegiao!A:ZZ', 'tiktokAlcance!A:ZZ', 'pinterestGeral!A:ZZ', 'pinterestGenero!A:ZZ', 'pinterestIdade!A:ZZ', 'pinterestRegiao!A:ZZ', 'pinterestAlcance!A:ZZ', 'linkedinGeral!A:ZZ', 'linkedinRegiao!A:ZZ', 'linkedinAlcance!A:ZZ', 'GAGeral!A:ZZ']
19:12:13 INFO extract.sheets_fetcher › 🔍 range completo retornado pela API: metaGeral!A1:AB615
19:12:13 INFO extract.sheets_fetcher › 🔍 range completo retornado pela API: metaIdade!A1:Q2068
19:12:13 INFO extract.sheets_fetcher › 🔍 range completo retornado pela API: metaGenero!A1:T928
19:12:13 INFO extract.sheets_fetcher › 🔍 range completo retornado pela API: metaRegiao!A1:Q10390
19:12:13 INFO extract.sheets_fetcher › 🔍 range completo retornado pela API: metaAlcance!A1:N615
19:12:13 INFO extract.sheets_fetcher › 🔍 range

In [11]:
#9
# — Forçar recarregamento dos parâmetros BI em qualquer ponto do notebook —
from treat.bi_param_utils import BIParamLookup

# Zera o cache interno para que a próxima chamada a .df() refaça o carregamento
BIParamLookup._df = None
BIParamLookup._last_load = 0.0

In [12]:
#10 – Validar consistência de datas entre modelos

from treat.utils.validations import validate_consistent_dates_across_models

# Extrai apenas os DataFrames de destino
dest_dfs = {sheet: info["dest"] for sheet, info in results.items()}

# Roda a validação
df_inconsistencies = validate_consistent_dates_across_models(dest_dfs)

# Exibe resultados
if df_inconsistencies is not None and not df_inconsistencies.empty:
    display(df_inconsistencies)
else:
    logger.info("✅ Nenhuma divergência de start/end entre modelos.")

19:12:13 INFO treat.utils.validations › ✅ Nenhuma divergência de start/end entre modelos.
19:12:13 INFO __main__ › ✅ Nenhuma divergência de start/end entre modelos.


In [13]:
import gspread, google.auth
from pprint import pformat

creds, _ = google.auth.load_credentials_from_file(CREDS_PATH, scopes=[
    "https://www.googleapis.com/auth/spreadsheets.readonly"
])

gc = gspread.authorize(creds)
ss = gc.open_by_key(SPREADSHEET_ID)

stats = []
for ws in ss.worksheets():
    rows = ws.row_count
    cols = ws.col_count
    cells = rows * cols
    stats.append((cells, ws.title, rows, cols))

stats.sort(reverse=True)          # maiores primeiro
logger.info(pformat(stats[:40]))  # top 10 abas que mais ocupam células

19:12:14 INFO __main__ › [(3108195, 'modeloRegiao', 207213, 15),
 (1072425, 'modeloIdade', 71495, 15),
 (300510, 'LINKEDIN NEGOCIOS ALCANÇADOS', 11130, 27),
 (207444, 'modeloAlcance', 17287, 12),
 (193785, 'modeloGenero', 12919, 15),
 (192588, 'metaPontoControle', 16049, 12),
 (176630, 'metaRegiao', 10390, 17),
 (95602, 'linkedinRegiao', 7354, 13),
 (61160, 'GAGeral', 6116, 10),
 (60775, 'SupermetricsQueries', 935, 65),
 (60346, 'modeloGeral', 2321, 26),
 (47895, 'tiktokRegiao', 3193, 15),
 (36000, 'kawaiGeral', 1000, 36),
 (35156, 'metaIdade', 2068, 17),
 (33007, 'pinterestRegiao', 2539, 13),
 (32788, 'CONTEÚDO _MÍDIA', 1171, 28),
 (28000, 'LEGENDA', 1000, 28),
 (26000, 'sites', 1000, 26),
 (26000, 'googleRegiao', 1000, 26),
 (26000, 'googleIdade', 1000, 26),
 (26000, 'googleGenero', 1000, 26),
 (26000, 'cidade', 1000, 26),
 (26000, 'adserver', 1000, 26),
 (26000, 'MEDIUM', 1000, 26),
 (26000, 'GAEventos', 1000, 26),
 (26000, 'CATALISA', 1000, 26),
 (26000, 'Alright genero', 1000, 26)